# 📊 Phase 4 — Évaluation complète : IoU · Cartes d'anomalie · Démonstration
**Projet PFE — Détection d'Anomalies dans des Objets Industriels 3D**  
**Stagiaire :** Olivier OUEDRAOGO — SUPMTI ISI  
**Entreprise :** 3D SMART FACTORY, Mohammedia  

---
**⚠️ Prérequis : exécuter dans la même session que la Phase 3**  
Les variables suivantes doivent être en mémoire :
`extractor`, `knn_btf`, `pca`, `test_scores`, `test_labels`, `test_defects`, `test_samples`, `THRESHOLD`

**Objectifs de cette phase :**
- Générer les cartes d'anomalie visuelles (heatmap superposée sur RGB)
- Calculer l'IoU entre masque prédit et masque ground truth
- Analyser les performances par type de défaut
- Produire un bilan final complet de toutes les métriques
- Démonstration interactive sur une pièce au choix

## 1. Setup — vérification de la session Phase 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn.functional as F
import tifffile
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import normalize as sk_normalize

# Vérification que la session Phase 3 est bien active
checks = {
    'extractor'    : 'extractor' in dir(),
    'knn_btf'      : 'knn_btf' in dir(),
    'pca'          : 'pca' in dir(),
    'test_scores'  : 'test_scores' in dir(),
    'test_samples' : 'test_samples' in dir(),
    'THRESHOLD'    : 'THRESHOLD' in dir(),
}

print('=== Vérification session Phase 3 ===')
all_ok = True
for name, ok in checks.items():
    status = '✅' if ok else '❌'
    print(f'  {status}  {name}')
    if not ok:
        all_ok = False

if all_ok:
    print('\n✅ Session Phase 3 active — Phase 4 prête à démarrer')
    print(f'   Test scores : {len(test_scores)} pièces')
    print(f'   Seuil       : {THRESHOLD:.6f}')
    print(f'   DEVICE      : {DEVICE}')
else:
    print('\n❌ Session expirée — relancez le notebook Phase 3 complet')

## 2. Cartes d'anomalie — heatmap patches sur image RGB

Pour chaque pièce, on calcule la distance KNN de chaque patch (28×28 positions)  
et on projette cette carte d'erreur sur l'image RGB en superposition colorée.  
**Rouge** = zone très anormale · **Vert** = zone normale

In [ ]:
def compute_anomaly_map(rgb_tensor, depth_tensor, extractor, knn, pca, device):
    """
    Calcule la carte d'anomalie spatiale pour une pièce.
    Retourne une heatmap (28, 28) avec le score d'anomalie par patch.
    """
    extractor.eval()
    with torch.no_grad():
        rgb   = rgb_tensor.unsqueeze(0).to(device)
        depth = depth_tensor.unsqueeze(0).to(device)

        r1, r2, r3 = extractor(rgb)
        d1, d2, d3 = extractor(depth)

        r1_r = F.interpolate(r1, size=(28,28), mode='bilinear', align_corners=False)
        r3_r = F.interpolate(r3, size=(28,28), mode='bilinear', align_corners=False)
        d1_r = F.interpolate(d1, size=(28,28), mode='bilinear', align_corners=False)
        d3_r = F.interpolate(d3, size=(28,28), mode='bilinear', align_corners=False)

        combined = torch.cat([r1_r, r2, r3_r, d1_r, d2, d3_r], dim=1)  # (1, 896, 28, 28)
        _, C, H, W = combined.shape

        # Extraire tous les patches : (H*W, C)
        patches = combined[0].permute(1,2,0).reshape(-1, C).cpu().numpy()
        patches = sk_normalize(patches)
        patches_pca = pca.transform(patches)           # (H*W, 128)

        # Distance KNN pour chaque patch
        dists, _ = knn.kneighbors(patches_pca)         # (H*W, 9)
        scores   = dists.mean(axis=1)                  # (H*W,)
        heatmap  = scores.reshape(H, W)                # (28, 28)

    return heatmap


def load_sample_tensors(sample, transform):
    """Charge RGB + Depth tensors pour un échantillon."""
    rgb_path = str(sample['xyz']).replace('/xyz/', '/rgb/').replace('.tiff', '.png')
    try:
        img        = Image.open(rgb_path).convert('RGB')
        img_tensor = transform(img)
        img_rgb    = np.array(img.resize((224, 224)))
    except:
        img_tensor = torch.zeros(3, 224, 224)
        img_rgb    = np.zeros((224, 224, 3), dtype=np.uint8)

    try:
        xyz_map  = tifffile.imread(str(sample['xyz']))
        z_map    = xyz_map[:,:,2].astype(np.float32)
        z_map    = (z_map - z_map.min()) / (z_map.max() - z_map.min() + 1e-8)
        z_tensor = transform(Image.fromarray((z_map*255).astype('uint8')).convert('RGB'))
    except:
        z_tensor = torch.zeros(3, 224, 224)

    gt_mask = None
    if sample['gt'] is not None:
        gt_img  = plt.imread(str(sample['gt']))
        if gt_img.ndim == 3:
            gt_img = gt_img[:,:,0]
        gt_mask = (gt_img > 0.5).astype(np.float32)

    return img_tensor, z_tensor, img_rgb, gt_mask


print('✅ Fonctions de carte d\'anomalie définies')

In [ ]:
import torchvision.transforms as T

rgb_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Visualisation : un exemple par type de défaut
defect_types = ['good', 'crack', 'hole', 'contamination', 'combined']
fig, axes    = plt.subplots(len(defect_types), 3,
                             figsize=(12, 4 * len(defect_types)))

col_titles = ['Image RGB', 'Carte d\'anomalie', 'RGB + Heatmap superposée']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight='bold')

for row, dtype in enumerate(defect_types):
    # Trouver un échantillon de ce type
    sample = next((s for s in test_samples if s['defect_type'] == dtype), None)
    if sample is None:
        continue

    img_t, depth_t, img_rgb, gt_mask = load_sample_tensors(sample, rgb_transform)
    heatmap = compute_anomaly_map(img_t, depth_t, extractor, knn_btf, pca, DEVICE)

    # Upscale heatmap → 224×224
    heatmap_up = np.array(Image.fromarray(heatmap).resize((224, 224), Image.BILINEAR))

    # Normalisation 0-1
    hmin, hmax = heatmap_up.min(), heatmap_up.max()
    heatmap_norm = (heatmap_up - hmin) / (hmax - hmin + 1e-8)

    # Score de la pièce
    idx_sample = next(i for i, s in enumerate(test_samples)
                      if str(s['xyz']) == str(sample['xyz']))
    score      = test_scores[idx_sample]
    is_defect  = score > THRESHOLD
    color_title = 'red' if dtype != 'good' else 'green'
    decision    = '⚠️ DÉFAUT' if is_defect else '✅ NORMAL'

    # Colonne 1 : RGB
    axes[row][0].imshow(img_rgb)
    axes[row][0].set_title(f'{dtype}  —  {decision}  (score={score:.4f})',
                            color=color_title, fontsize=9)
    axes[row][0].axis('off')

    # Colonne 2 : Heatmap seule
    im = axes[row][1].imshow(heatmap_norm, cmap='RdYlGn_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[row][1], fraction=0.046)
    axes[row][1].set_title('Score anomalie (rouge=élevé)', fontsize=9)
    axes[row][1].axis('off')

    # Colonne 3 : Superposition
    axes[row][2].imshow(img_rgb)
    axes[row][2].imshow(heatmap_norm, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    if gt_mask is not None:
        gt_up = np.array(Image.fromarray(gt_mask).resize((224, 224), Image.NEAREST))
        axes[row][2].contour(gt_up, levels=[0.5], colors='white', linewidths=1.5)
        axes[row][2].set_title('Superposé (contour blanc=GT)', fontsize=9)
    else:
        axes[row][2].set_title('Superposé', fontsize=9)
    axes[row][2].axis('off')

plt.suptitle('Cartes d\'anomalie BTF — bagel (contour blanc = vérité terrain)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/anomaly_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cartes sauvegardées : /content/anomaly_maps.png')

## 3. Calcul de l'IoU — masque prédit vs masque Ground Truth

L'IoU (Intersection over Union) mesure la précision de localisation :  
- On seuille la heatmap pour obtenir un masque binaire prédit
- On compare pixel par pixel avec le masque GT annoté manuellement
- IoU = |Intersection| / |Union|

In [ ]:
def compute_iou(pred_mask, gt_mask):
    """
    Calcule l'IoU entre masque prédit et masque GT.
    pred_mask, gt_mask : tableaux binaires de même taille.
    """
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    intersection = (pred & gt).sum()
    union        = (pred | gt).sum()
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union


def compute_all_ious(test_samples, test_scores, threshold,
                     extractor, knn, pca, transform, device):
    """
    Calcule l'IoU pour toutes les pièces défectueuses du test.
    Le seuil de heatmap est calibré automatiquement (percentile 90).
    """
    results = []
    extractor.eval()

    for i, sample in enumerate(test_samples):
        if sample['gt'] is None:
            continue

        img_t, depth_t, _, gt_mask = load_sample_tensors(sample, transform)
        heatmap = compute_anomaly_map(img_t, depth_t, extractor, knn, pca, device)

        # Upscale heatmap à la taille du masque GT
        gt_h, gt_w = gt_mask.shape
        heatmap_up = np.array(
            Image.fromarray(heatmap).resize((gt_w, gt_h), Image.BILINEAR))

        # Seuil adaptatif : percentile 90 de la heatmap
        heatmap_thresh = np.percentile(heatmap_up, 90)
        pred_mask      = (heatmap_up > heatmap_thresh).astype(np.float32)

        iou = compute_iou(pred_mask, gt_mask)
        results.append({
            'defect_type': sample['defect_type'],
            'iou':         iou,
            'score':       test_scores[i] if i < len(test_scores) else 0,
            'path':        str(sample['xyz'])
        })

        if (i+1) % 20 == 0:
            print(f'  {i+1}/{len(test_samples)} pièces traitées...')

    return results


print('⏳ Calcul IoU sur toutes les pièces défectueuses...')
iou_results = compute_all_ious(
    test_samples, test_scores, THRESHOLD,
    extractor, knn_btf, pca, rgb_transform, DEVICE
)

print(f'\n✅ IoU calculé pour {len(iou_results)} pièces défectueuses')

## 4. Analyse IoU par type de défaut

In [ ]:
import pandas as pd

df_iou = pd.DataFrame(iou_results)

print('=== IoU par type de défaut ===')
print(f'  (objectif cahier des charges : IoU > 0.60)\n')

iou_by_type = {}
for dtype in ['crack', 'hole', 'contamination', 'combined']:
    subset = df_iou[df_iou['defect_type'] == dtype]['iou']
    if len(subset) > 0:
        mean_iou = subset.mean()
        iou_by_type[dtype] = mean_iou
        status = '✅' if mean_iou >= 0.60 else '⚠️ '
        bar    = '█' * int(mean_iou * 20)
        print(f'  {status}  {dtype:15s} → IoU={mean_iou:.4f}  n={len(subset)}  {bar}')

global_iou = df_iou['iou'].mean()
status_global = '✅' if global_iou >= 0.60 else '⚠️ '
print(f'\n  {status_global}  IoU global moyen : {global_iou:.4f}  (objectif > 0.60)')

# Visualisation IoU
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barplot IoU par type
types  = list(iou_by_type.keys())
values = list(iou_by_type.values())
colors = ['#2ecc71' if v >= 0.6 else '#e74c3c' for v in values]
axes[0].bar(types, values, color=colors, edgecolor='white', alpha=0.85)
axes[0].axhline(0.60, color='black', linestyle='--', linewidth=2,
                label='Objectif IoU=0.60')
axes[0].set_ylim(0, 1)
axes[0].set_title('IoU moyen par type de défaut')
axes[0].set_xlabel('Type de défaut')
axes[0].set_ylabel('IoU')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Distribution IoU (violinplot)
data_by_type = [df_iou[df_iou['defect_type']==d]['iou'].values
                for d in types if len(df_iou[df_iou['defect_type']==d]) > 0]
valid_types  = [d for d in types if len(df_iou[df_iou['defect_type']==d]) > 0]
if data_by_type:
    bp = axes[1].boxplot(data_by_type, labels=valid_types, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1].axhline(0.60, color='black', linestyle='--', linewidth=2)
    axes[1].set_ylim(0, 1)
    axes[1].set_title('Distribution IoU par type')
    axes[1].set_ylabel('IoU')
    axes[1].grid(alpha=0.3, axis='y')

plt.suptitle(f'IoU — Localisation des anomalies | Moyen={global_iou:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/iou_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure IoU sauvegardée : /content/iou_results.png')

## 5. Bilan final — toutes métriques réunies

Récapitulatif complet de toutes les métriques du projet.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

test_preds = (test_scores > THRESHOLD).astype(int)
auc  = roc_auc_score(test_labels, test_scores)
prec = precision_score(test_labels, test_preds, zero_division=0)
rec  = recall_score(test_labels, test_preds, zero_division=0)
f1   = f1_score(test_labels, test_preds, zero_division=0)

print('╔══════════════════════════════════════════════════════════╗')
print('║         BILAN FINAL — PHASE 4 — bagel                   ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  DÉTECTION GLOBALE                                       ║')
print(f'║   AUC-ROC    : {auc:.4f}   objectif > 0.85  {"✅" if auc>=0.85 else "❌"}          ║')
print(f'║   F1-Score   : {f1:.4f}   objectif > 0.80  {"✅" if f1>=0.80 else "❌"}          ║')
print(f'║   Précision  : {prec:.4f}   objectif > 0.80  {"✅" if prec>=0.80 else "❌"}          ║')
print(f'║   Rappel     : {rec:.4f}   objectif > 0.80  {"✅" if rec>=0.80 else "❌"}          ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  LOCALISATION (IoU)                                      ║')
for dtype, iou_val in iou_by_type.items():
    status = '✅' if iou_val >= 0.60 else '⚠️ '
    print(f'║   {status} {dtype:15s} : IoU={iou_val:.4f}               ║')
print(f'║   IoU global moyen   : {global_iou:.4f}   objectif > 0.60  {"✅" if global_iou>=0.60 else "⚠️ "}          ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  VALIDATION DU MODÈLE                                    ║')
print('║   ✅  Anti-fuite      : 0 chevauchement train/test       ║')
print('║   ✅  Robustesse      : image aléatoire score 2x normaux  ║')
print('║   ✅  Sémantique      : AUC varie par type de défaut      ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  APPROCHE RETENUE                                        ║')
print('║   BTF (Back to Feature) — ResNet18 ImageNet              ║')
print('║   Modalités : RGB + carte de profondeur Z                ║')
print('║   Features  : multi-échelle (L1+L2+L3) → PCA → KNN      ║')
print('╚══════════════════════════════════════════════════════════╝')

# Sauvegarder les métriques en JSON
import json, os
metrics = {
    'detection': {
        'auc_roc':   round(auc,  4),
        'f1_score':  round(f1,   4),
        'precision': round(prec, 4),
        'recall':    round(rec,  4),
        'threshold': round(THRESHOLD, 6)
    },
    'localization': {
        'iou_global': round(global_iou, 4),
        'iou_by_type': {k: round(v, 4) for k, v in iou_by_type.items()}
    },
    'model': {
        'approach':    'BTF RGB+Depth',
        'backbone':    'ResNet18 ImageNet',
        'modalities':  ['RGB', 'Depth Z'],
        'n_train':     len(train_samples),
        'category':    CATEGORY
    }
}

METRICS_PATH = '/content/drive/MyDrive/Detection_Anomalie_3D/results'
os.makedirs(METRICS_PATH, exist_ok=True)
with open(f'{METRICS_PATH}/metrics_final.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\n✅ Métriques sauvegardées : {METRICS_PATH}/metrics_final.json')

## 6. Démonstration interactive

Choisissez une pièce par son index dans le test (0 à 109) et le système  
affiche : décision, score, carte d'anomalie et comparaison avec le masque GT.

In [ ]:
def demo_piece(sample_idx, test_samples, test_scores, threshold,
               extractor, knn, pca, transform, device):
    """
    Démonstration complète sur une pièce au choix.
    Affiche : décision · score · heatmap · comparaison GT
    """
    if sample_idx >= len(test_samples):
        print(f'Index invalide — max : {len(test_samples)-1}')
        return

    sample = test_samples[sample_idx]
    score  = test_scores[sample_idx]
    label  = sample['defect_type']
    is_def = score > threshold

    print('=' * 50)
    print(f'  DÉMONSTRATION — Pièce #{sample_idx}')
    print('=' * 50)
    print(f'  Type réel    : {label}')
    print(f'  Score        : {score:.6f}')
    print(f'  Seuil        : {threshold:.6f}')
    print(f'  Décision     : {"⚠️  DÉFAUT DÉTECTÉ" if is_def else "✅ PIÈCE NORMALE"}')

    correct = (is_def and label != 'good') or (not is_def and label == 'good')
    print(f'  Résultat     : {"✅ CORRECT" if correct else "❌ ERREUR"}')
    print('=' * 50)

    img_t, depth_t, img_rgb, gt_mask = load_sample_tensors(sample, transform)
    heatmap = compute_anomaly_map(img_t, depth_t, extractor, knn, pca, device)
    heatmap_up = np.array(Image.fromarray(heatmap).resize((224, 224), Image.BILINEAR))
    hmin, hmax = heatmap_up.min(), heatmap_up.max()
    heatmap_norm = (heatmap_up - hmin) / (hmax - hmin + 1e-8)

    n_cols = 4 if gt_mask is not None else 3
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4.5))

    # RGB
    axes[0].imshow(img_rgb)
    color = 'red' if label != 'good' else 'green'
    axes[0].set_title(f'RGB — {label}', color=color, fontsize=10)
    axes[0].axis('off')

    # Heatmap
    im = axes[1].imshow(heatmap_norm, cmap='RdYlGn_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    axes[1].set_title('Carte d\'anomalie', fontsize=10)
    axes[1].axis('off')

    # Superposition
    axes[2].imshow(img_rgb)
    axes[2].imshow(heatmap_norm, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    if gt_mask is not None:
        gt_up = np.array(Image.fromarray(gt_mask).resize((224, 224), Image.NEAREST))
        axes[2].contour(gt_up, levels=[0.5], colors='white', linewidths=2)
    axes[2].set_title('Superposé (blanc=GT)', fontsize=10)
    axes[2].axis('off')

    # GT si disponible
    if gt_mask is not None and n_cols == 4:
        gt_up = np.array(Image.fromarray(gt_mask).resize((224, 224), Image.NEAREST))
        axes[3].imshow(gt_up, cmap='Reds')
        iou_val = compute_iou(
            (heatmap_norm > np.percentile(heatmap_norm, 90)).astype(float), gt_mask)
        axes[3].set_title(f'Masque GT\nIoU={iou_val:.4f}', fontsize=10)
        axes[3].axis('off')

    decision_txt = '⚠️ DÉFAUT' if is_def else '✅ NORMAL'
    correct_txt  = '✅ CORRECT' if correct else '❌ ERREUR'
    plt.suptitle(
        f'Pièce #{sample_idx} — {label} | Score={score:.4f} | {decision_txt} | {correct_txt}',
        fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'/content/demo_piece_{sample_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()


# ─── MODIFIEZ CET INDEX POUR CHOISIR LA PIÈCE ────────────────────
SAMPLE_IDX = 0   # 0 à 109
# ─────────────────────────────────────────────────────────────────

# Afficher la liste des pièces disponibles
print('Pièces disponibles dans le test :')
for i, s in enumerate(test_samples[:20]):
    score = test_scores[i] if i < len(test_scores) else 0
    print(f'  [{i:3d}]  {s["defect_type"]:15s}  score={score:.4f}')
print(f'  ... (110 pièces total)\n')

demo_piece(SAMPLE_IDX, test_samples, test_scores, THRESHOLD,
           extractor, knn_btf, pca, rgb_transform, DEVICE)

In [ ]:
# ─── Lancez cette cellule pour tester d'autres pièces ────────────
# Modifiez SAMPLE_IDX et relancez

SAMPLE_IDX = 10   # Changez cet index

demo_piece(SAMPLE_IDX, test_samples, test_scores, THRESHOLD,
           extractor, knn_btf, pca, rgb_transform, DEVICE)

## Bilan Phase 4

**Phase 4 terminée.** Les livrables produits sont :

| Livrable | Fichier |
|---|---|
| Cartes d'anomalie (5 types) | `/content/anomaly_maps.png` |
| Résultats IoU | `/content/iou_results.png` |
| Métriques JSON | `Drive/results/metrics_final.json` |
| Démos individuelles | `/content/demo_piece_N.png` |

**Prochaine étape — Phase 5 : Déploiement & soutenance**
- Interface de démonstration live
- Rapport final complet
- Préparation slides de soutenance